# ⚙️ Requisitos e instalación

**Python 3.11 o superior** (probado con 3.13). Las dependencias se instalan en un entorno virtual.

### 1. Crear el entorno e instalar las dependencias

Con **uv** (recomendado):
```powershell
uv venv .venv
uv pip install pandas google-genai pydantic python-dotenv ipykernel
```

O con **pip**:
```powershell
python -m venv .venv
.\.venv\Scripts\activate        # Linux/macOS: source .venv/bin/activate
pip install pandas google-genai pydantic python-dotenv ipykernel
```

| Paquete | Para qué |
|---|---|
| `pandas` | Carga y manipulación de los CSV |
| `google-genai` | SDK oficial de Gemini |
| `pydantic` | Esquemas de *structured output* y validación del JSON devuelto |
| `python-dotenv` | Cargar la API key desde un fichero `.env` |
| `ipykernel` | Ejecutar el notebook con el entorno virtual como kernel |

### 2. Seleccionar el kernel

En VS Code: *Select Kernel* → *Python Environments* → el `python.exe` de `.venv`. Si el entorno está fuera de esta carpeta (por ejemplo un nivel por encima), selecciónalo con *Enter interpreter path…*.

### 3. Configurar la API key de Google Gemini

1. Consigue una clave gratuita en [Google AI Studio](https://aistudio.google.com/apikey).
2. Crea un fichero `.env` con esta línea:
   ```
   GOOGLE_API_KEY=tu_clave
   ```
   Puede ir en la carpeta del notebook o en cualquier carpeta superior: el notebook lo busca hacia arriba. **No lo subas a Git**; ya está incluido en `.gitignore`.
   También vale definir `GOOGLE_API_KEY` como variable de entorno del sistema.
3. **En Google Colab:** guarda la clave en *Secrets* (icono 🔑 del panel izquierdo) con el nombre `GOOGLE_API_KEY` y da acceso al notebook.

**Si no se encuentra la clave por ninguna de esas vías, el notebook la pedirá por teclado** al ejecutar la celda de configuración. Se introduce con `getpass`, así que no se muestra en pantalla ni queda guardada en el notebook.

### 4. Datos

Los CSV deben estar en la misma carpeta que el notebook: `videogames_reviews.csv` (y, opcionalmente, el ejemplo `analisis_videojuegos_resultados.csv`).

# **Ejercicio 2: Entregable Final (videogames_reviews.csv)**

Vamos a trabajar con un dataframe que contiene información sobre reseñas de videojuegos. Cada fila del dataframe contiene la información de una reseña, y las columnas son las siguientes:
- Contenido: texto de la reseña
- Valoración: Recomendado o No recomendado
- Recomendado binario: 1 si la valoración es Recomendado, 0 si la valoración es No recomendado

El objetivo de este ejercicio es conseguir extraer información valiosa de las reseñas de un dataset más 'crudo' o que necesita un preprocesado mayor que el del ejercicio 1.

1. **Filtramos por las 100 reseñas con el contenido más largo**. Esto ya os lo doy hecho más abajo.
2. En el DataFrame resultante, habremos obtenido muchas filas en las que realmente el contenido de la reseña no es nada relevante. **Instruir vía Prompting a un LLM para que filtre las filas del DataFrame que de verdad aportan información de la que se pueden extraer insights relevantes**
3. **Paso final:** extraer información relevante de este último dataframe doblemente filtrado usando un segundo LLM. Por ejemplo, un json final como este:

    ```json
    {
    "Id": "{{id}}",
    "SentimientoGeneral": "Positivo|Negativo|Neutral",
    "AspectosPositivos": "[lista de aspectos positivos mencionados]",
    "AspectosNegativos": "[lista de aspectos negativos mencionados]",
    "Dificultad": "Demasiado Fácil|Fácil|Equilibrado|Difícil|Demasiado Difícil|No Mencionado",
    "Recomendado": "{{recomendado}}"
    }
    ```

**Consideraciones adicionales:**

- También tienes un csv de ejemplo de resultado final (analisis_videojuegos_resultados.csv) donde se han extraído otras entidades y se ha entregado otro formato. **Esto es a libre elección, pero un mínimo de 3 entidades extraídas** por el segundo LLM.
- Los modelos suelen ser buenos realizando tareas de una en una. Por eso separamos en dos pasos el filtrado de contenido relevante y el de extracción de ese contenido.
- Puedes usar cualquier proveedor y canal: vía API (como Groq/Gemini) o en local vía HuggingFce teniendo en cuenta las limitaciones y ventajas de cada uno, rate limits del modelo elegido, tamaño...
- Tener en cuenta qué cantidad de tokens/información/tareas le estamos pasando al LLM en cada llamada comparándolo con los límites que tenemos para el LLM elegido
- Se valorará **justificar cada elección** como: cada time.stop() añadido, cada lote de filas de DataFrame que pasemos en cada llamada... etc ajustándonos como digo a esos límites que tenemos.

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('videogames_reviews.csv')
data

,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0
...,...,...,...,...
19995,19995,si,Recomendado,1
19996,19996,cojonudo,Recomendado,1
19997,19997,reostia historia guapisima graficos impresiona...,Recomendado,1
19998,19998,basicamente sublime obra maestra,Recomendado,1


In [3]:
# Análisis exploratorio básico
print("Información básica del dataframe:")
print(f"Forma del dataframe: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print("\nTipos de datos:")
print(data.dtypes)
print("\nPrimeras filas:")
data.head()

Información básica del dataframe:
Forma del dataframe: (20000, 4)
Columnas: ['Unnamed: 0', 'Contenido', 'Valoración', 'Recomendado_binario']

Tipos de datos:
Unnamed: 0             int64
Contenido                str
Valoración               str
Recomendado_binario    int64
dtype: object

Primeras filas:


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0


In [4]:
# Análisis de valores nulos y únicos
print("Valores nulos por columna:")
print(data.isnull().sum())
print("\nValores únicos en 'Valoración':")
print(data['Valoración'].value_counts())
print("\nEstadísticas de la columna 'Recomendado_binario':")
print(data['Recomendado_binario'].value_counts())

Valores nulos por columna:
Unnamed: 0               0
Contenido              288
Valoración               0
Recomendado_binario      0
dtype: int64

Valores únicos en 'Valoración':
Valoración
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64

Estadísticas de la columna 'Recomendado_binario':
Recomendado_binario
0    10000
1    10000
Name: count, dtype: int64


## **PASO 1**
Obtener los 100 comentarios con más texto para análisis posterior.

> **Único cambio respecto a la plantilla:** en vez de eliminar `Unnamed: 0`, la renombro a `indice_original`. Así cada resultado final se puede rastrear hasta su fila en el CSV original.

In [5]:
# Preprocesamiento simplificado: Top 100 comentarios con más texto
print("Preprocesando datos para obtener los 100 comentarios más largos...")

# Eliminar filas con contenido nulo
df_limpio = data.dropna(subset=['Contenido']).copy()
print(f"Comentarios válidos (sin nulos): {len(df_limpio)}")

# Calcular longitud de caracteres
df_limpio['longitud_caracteres'] = df_limpio['Contenido'].str.len()

# Seleccionar los 100 comentarios más largos
df_top_100 = df_limpio.nlargest(100, 'longitud_caracteres').reset_index(drop=True)

# Conservar el índice original para trazabilidad (la plantilla lo eliminaba)
if 'Unnamed: 0' in df_top_100.columns:
    df_top_100 = df_top_100.rename(columns={'Unnamed: 0': 'indice_original'})

print(f"\nDataset final: {len(df_top_100)} comentarios")
print(f"Longitud promedio: {df_top_100['longitud_caracteres'].mean():.1f} caracteres")
print(f"Longitud mínima: {df_top_100['longitud_caracteres'].min()} caracteres")
print(f"Longitud máxima: {df_top_100['longitud_caracteres'].max()} caracteres")

print("\nDistribución de valoraciones en el top 100:")
print(df_top_100['Valoración'].value_counts())

# Mostrar el dataframe resultante
df_top_100

Preprocesando datos para obtener los 100 comentarios más largos...
Comentarios válidos (sin nulos): 19712

Dataset final: 100 comentarios
Longitud promedio: 6772.1 caracteres
Longitud mínima: 5421 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 100:
Valoración
No recomendado    65
Recomendado       35
Name: count, dtype: int64


,indice_original,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,19502,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,17660,this was probably my first preorder i felt tha...,Recomendado,1,7662
2,4187,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,7187,oh well admittedly its difficult for to write ...,No recomendado,0,7638
4,5415,i know many will handwave away any criticisms ...,No recomendado,0,7609
...,...,...,...,...,...
95,12474,4 febrero i was not seated in the glorious hyp...,Recomendado,1,5607
96,11186,cyberpunk 2077cybermeme 2077what can there be ...,Recomendado,1,5573
97,14895,brujo geralt riviatras recuperar memoria pasad...,Recomendado,1,5516
98,1962,after hearing all the fuss about skyrim for ma...,No recomendado,0,5458


### observaciones sobre los datos y deduplicado (sin LLM)

Al revisar el top 100 se ven tres cosas que condicionan el diseño de los prompts:

1. **El texto ya viene normalizado:** minúsculas, **sin signos de puntuación**, palabras a veces pegadas (`introductionelden ring`) y, en las reseñas en español, sin palabras vacías (`actualmente recomiendo juego contaba requisitos…`). El LLM tiene que saberlo para no descartar reseñas por estar "mal escritas", y los ejemplos *few-shot* deben imitar este estilo.
2. **La mayoría de reseñas están en inglés**, con algunas en español. Los modelos actuales entienden ambos idiomas; se pide la salida en español para que sea homogénea.
3. **Hay reseñas duplicadas exactas** (la misma reseña copiada varias veces). Mandarlas al LLM gastaría llamadas y tokens en analizar el mismo texto. Se eliminan con pandas, que es gratis y determinista, **antes** de cualquier llamada.

In [6]:
print("Duplicados exactos en el top 100:", df_top_100.duplicated(subset="Contenido").sum())

df_top = df_top_100.drop_duplicates(subset="Contenido").reset_index(drop=True)
df_top.index.name = "id"
print("Reseñas únicas que pasan al LLM:", len(df_top))
print(df_top["Valoración"].value_counts())

Duplicados exactos en el top 100: 20
Reseñas únicas que pasan al LLM: 80
Valoración
No recomendado    49
Recomendado       31
Name: count, dtype: int64


---
## Proveedor y modelos

Pipeline con **dos LLMs**, cada uno con una única tarea:

```
df_top (reseñas únicas)
   ▼
LLM 1 · FILTRO      gemini-3.5-flash-lite  →  ¿aporta información? (+ motivo)  →  df_relevantes
   ▼
LLM 2 · EXTRACCIÓN  gemini-3-flash-preview →  JSON estructurado por reseña     →  df_final / CSV
```

| | LLM 1 · Filtro | LLM 2 · Extracción |
|---|---|---|
| Modelo principal | `gemini-3.5-flash-lite` | `gemini-3-flash-preview` |
| Reservas (en orden) | `gemini-3.1-flash-lite`, `gemini-flash-lite-latest` | `gemini-flash-latest` (alias de `gemini-3.8-flash` a 2026-11-03), `gemini-3.5-flash-lite` |
| Tarea | Clasificación binaria + motivo | Extracción de muchos campos, con matices |
| Por qué | Tarea sencilla: prima coste y velocidad, con más RPM y RPD en la capa gratuita | Hace falta más comprensión: ironía, reseñas mixtas, texto sin puntuación, jerga *gamer* |
| Temperatura | 0 | 0 |

**Por qué Gemini:**
- *Structured output* nativo: se le pasa un esquema Pydantic y el JSON llega validado. Esto importa: en el CSV de ejemplo, el campo `sentiment` aparece como `neutro`, `neutral`, `mixed` o `negative` mezclados. Con valores cerrados (`Literal`) eso no puede pasar.
- Ventana de contexto de 1M tokens: las reseñas largas (~1.360 tokens de media cada una) caben en lotes sin problema.
- Capa gratuita suficiente para ~100 reseñas y dos modelos con la misma API key.

**Disponibilidad real de los modelos** (comprobada con la API key el 2026-11-03):

| Modelo | Resultado |
|---|---|
| `gemini-2.5-flash`, `gemini-2.5-flash-lite` | **404**: ya no disponibles para cuentas nuevas |
| `gemini-3.8-flash`, `gemini-flash-latest` | **503**: saturados (*high demand*); error temporal pero frecuente |
| `gemini-3.5-flash-lite`, `gemini-3.1-flash-lite`, `gemini-flash-lite-latest` | OK, ~1 s por respuesta |
| `gemini-3.5-flash` | OK, ~10 s por respuesta (razona antes de contestar) |

**Segunda comprobación, el mismo día a las 21:29**, tras una primera ejecución completa del notebook:

| Modelo | Resultado |
|---|---|
| `gemini-3.5-flash`, `gemini-3.6-flash`, `gemini-pro-latest`, `gemini-3.1-pro-preview` | **429**: cuota diaria agotada |
| `gemini-3.7-flash` | **503**: saturado |
| `gemini-3.8-flash` / `gemini-flash-latest` (mismo modelo) | OK, pero con latencias muy distintas: 117 s y 13 s en dos llamadas casi simultáneas |
| `gemini-3-flash-preview` | OK, ~11 s por respuesta |

#### Un mismo modelo, dos latencias: la saturación como causa probable

`gemini-flash-latest` es un **alias** que Google apunta a la última versión de Gemini Flash; a fecha 2026-11-03, `gemini-3.8-flash`. Por tanto, en las pruebas anteriores **son el mismo modelo**, y aun así:

- En la primera comprobación, **los dos devolvieron 503 a la vez**, como cabe esperar de un mismo modelo saturado.
- En la segunda, con la misma petición, **uno tardó 117 s y el otro 13 s**, y devolvieron **la misma respuesta** (`sentimiento=Mixto`, `jugabilidad=Mala`, `horas=60`).
- En las ejecuciones del notebook, la latencia de la extracción fue muy irregular: en la primera, un lote tardó 36 s y el siguiente ~5 min (reintentos por 503); en otra, el primer lote tardó ~13 min en completarse y los siguientes ~15 s cada uno; y en la penúltima, entre dos lotes consecutivos pasaron **21 minutos** mientras el iterador volvía a probar `gemini-flash-latest`, todavía saturado.

**Conjetura:** la diferencia no se debe al modelo, sino a la **latencia variable por saturación**. Cuando hay mucha demanda, el servicio de Google rechaza la petición (503) o la deja en cola, y la respuesta tarda mucho más. Es una conjetura, no algo comprobado: la API no indica a qué versión apunta el alias en cada llamada ni cuánto tiempo pasa la petición en cola, y solo tenemos unas pocas mediciones.

**Consecuencias para el diseño:**
- **Una medición puntual de latencia no sirve para elegir modelo.** `gemini-flash-latest` no es "más rápido" que `gemini-3.8-flash`: se eligió porque respondía en ese momento.
- **Los picos de saturación son impredecibles y transitorios.** Por eso hacen falta los reintentos con *backoff* y el cortacircuitos, que aparta 5 minutos a un modelo saturado en vez de insistir lote tras lote.
- **Usar el alias tiene un coste en reproducibilidad.** Si Google lo apunta a otra versión, el notebook usará otro modelo sin avisar. A cambio, no queda atado a una versión que pueda retirarse, como ya pasó con `gemini-2.5-flash` (404).

#### Por qué se cambió dos veces el modelo de extracción

1. **`gemini-3.5-flash` → `gemini-flash-latest`.** En la primera ejecución completa, `gemini-3.5-flash` devolvió **429 (cuota agotada)** a partir del lote 3, y el iterador pasó a `gemini-3.5-flash-lite`, el mismo modelo que filtra.
2. **`gemini-flash-latest` → `gemini-3-flash-preview`.** En la siguiente ejecución, `gemini-flash-latest` también devolvió **429** tras 3 lotes, pese a que su cuota nominal es de 1.500 peticiones/día. El iterador pasó a `gemini-3-flash-preview`, que procesó sin problemas los 13 lotes restantes.

Para intentar que la extracción fuera **homogénea, hecha por un único modelo y distinto del filtro**, se fijó `gemini-3-flash-preview` como modelo principal y se repitieron solo los 3 lotes que había procesado `gemini-flash-latest` (la caché conservó todo lo demás).

#### Resultado final: extracción no homogénea

| Lotes de extracción | Reseñas | Modelo que los procesó |
|---|---|---|
| 1-2 y 4-16 (15 lotes) | 74 | `gemini-3-flash-preview` |
| 3 (1 lote) | 5 (ids 11 a 15) | `gemini-3.5-flash-lite` |

En la repetición, el lote 3 recibió **429** de `gemini-3-flash-preview` y también de `gemini-flash-latest`, y el iterador lo resolvió con la última reserva, `gemini-3.5-flash-lite`, que es el mismo modelo que filtra. **La extracción no es homogénea por la saturación y los límites de cuota del servicio (errores 503 y 429), no por una decisión de diseño.** Se decidió no seguir reintentando para no depender de cuándo se liberase la cuota.

Esto es precisamente lo que justifica la **llamada multimodelo**: en un servicio gratuito sujeto a saturación, el modelo disponible puede cambiar **a mitad de ejecución**, y el pipeline tiene que poder continuar con otro en vez de detenerse. El precio es que el resultado puede mezclar modelos. Por eso el notebook **muestra cada error y cada cambio de modelo** en la salida de la celda de extracción, y esta tabla resume el resultado, para que la mezcla quede documentada y no oculta. Una ejecución posterior, con la cuota renovada, produciría una extracción hecha íntegramente con `gemini-3-flash-preview` sin cambiar el código.

*Nota: `contador_llamadas` solo cuenta las llamadas de la última ejecución; los lotes que vienen de la caché no aparecen en él.*

Por eso cada paso usa una **lista de 3 modelos en orden de preferencia**. Si el principal está saturado tras varios reintentos, o no está disponible, el notebook prueba automáticamente el siguiente.

*Alternativa descartada:* Groq (Llama 3.x) es muy rápido, pero en la capa gratuita tiene límites de tokens por minuto bajos (~6k TPM en varios modelos). Con reseñas de ~1.800 tokens, eso obligaría a lotes de 2-3 reseñas y a esperas largas entre llamadas.

La API key se lee de `GOOGLE_API_KEY` (variable de entorno o fichero `.env` en la carpeta del notebook o en cualquier carpeta superior); nunca va escrita en el notebook.

In [7]:
import os, json, time, random
from pathlib import Path
from typing import Literal, Optional
from collections import Counter

from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types, errors
from IPython.display import display

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))  # busca .env en la carpeta actual y sus padres
except ImportError:
    pass

def obtener_api_key() -> str:
    """Busca la clave en: variable de entorno / .env -> Secrets de Colab -> la pide por teclado."""
    clave = os.getenv("GOOGLE_API_KEY")
    if not clave:
        try:
            from google.colab import userdata   # solo existe en Google Colab
            clave = userdata.get("GOOGLE_API_KEY")
        except Exception:
            pass
    if not clave:
        from getpass import getpass             # la entrada no se muestra ni queda guardada en el notebook
        clave = getpass("Introduce tu GOOGLE_API_KEY: ").strip()
    if not clave:
        raise ValueError("No se ha proporcionado ninguna GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = clave        # no se vuelve a pedir en esta sesión
    return clave

API_KEY = obtener_api_key()
client = genai.Client(api_key=API_KEY)

# 3 modelos por paso, en orden de preferencia: si uno falla, se prueba el siguiente
# (ver los disponibles con: [m.name for m in client.models.list()])
MODELO_FILTRO = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-flash-lite-latest"]
MODELO_EXTRACCION = ["gemini-3-flash-preview", "gemini-flash-latest", "gemini-3.5-flash-lite"]

CACHE = Path("cache_llm"); CACHE.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 150)

## Gestión de limitaciones

### Límites de la API (capa gratuita)

| Modelo | Uso en el pipeline | RPM | TPM | RPD | Entrada / salida máx. |
|---|---|---|---|---|---|
| `gemini-3.5-flash-lite` | Filtro (principal) y reserva de la extracción | 30 | 1.000.000 | 1.500 | 1M / 65.536 tokens |
| `gemini-3-flash-preview` | Extracción (principal) | 15 | 1.000.000 | 1.500 | 1M / 65.536 tokens |
| `gemini-flash-latest` (alias de `gemini-3.8-flash` a 2026-11-03) | Reserva de la extracción | 15 | 1.000.000 | 1.500 | 1M / 65.536 tokens |

*RPM = peticiones por minuto; TPM = tokens por minuto; RPD = peticiones por día. Fuente de RPM, TPM y RPD: consulta al asistente Gemini el 2026-11-03 (valores nominales de la capa gratuita); límites de entrada y salida: API (`client.models.get`). `gemini-flash-latest` es un alias que Google actualiza a la última versión de Gemini Flash: si cambia, el notebook usará automáticamente la nueva.*

Los límites publicados no bastan: en la ejecución real aparecieron dos errores que no dependen del ritmo de llamadas:
- **503 (modelo saturado)**: lo provoca la demanda global de Google, no nuestro uso.
- **429 (cuota agotada)** en `gemini-3.5-flash` y en `gemini-flash-latest` tras muy pocas peticiones, lejos de su RPD nominal. Tampoco lo explica el TPM: cada lote envía ~7.000 tokens, menos del 1 % de 1.000.000. Los límites efectivos de la capa gratuita pueden ser menores que los publicados.

### Mecanismos

| Mecanismo | Qué resuelve | Parámetros | Justificación |
|---|---|---|---|
| `LimitadorRPM` | No superar el RPM | 12 RPM (filtro), 8 RPM (extracción) | Muy por debajo del límite. En la práctica casi no actúa: cada llamada tarda 5-15 s, así que el ritmo real ya es menor. Es una red de seguridad, no un `time.sleep()` fijo |
| Reintentos con *backoff* exponencial | Errores temporales (429, 500, 503) | 4 intentos: 5 s → 10 s → 20 s | Estrategia estándar ante límites por minuto: da tiempo a que se libere la ventana |
| Iterador de modelos | El modelo sigue fallando o no existe (404) | 3 modelos por paso | La cuota es independiente por modelo |
| Cortacircuitos | Repetir reintentos inútiles con un modelo saturado | Pausa de 5 min | Sin él, un lote llegó a tardar más de 6 min |
| JSON inválido = error temporal | Respuesta cortada por `max_output_tokens` | Reintento y, si persiste, siguiente modelo | Ocurrió con `gemini-flash-latest`: gastó tokens razonando y cortó el JSON |
| Validación de ids | El modelo omite filas de un lote | Reproceso individual | No se pierde ninguna reseña |
| Caché en disco | Repetir llamadas al reanudar | Un fichero por lote | No se gasta cuota diaria dos veces |

In [8]:
class LimitadorRPM:
    def __init__(self, rpm: int):
        self.intervalo = 60.0 / rpm
        self.ultimo = 0.0
    def esperar(self):
        restante = self.intervalo - (time.monotonic() - self.ultimo)
        if restante > 0:
            time.sleep(restante)
        self.ultimo = time.monotonic()

# Un limitador por paso del pipeline (clave: modelo principal de cada lista)
LIMITADORES = {MODELO_FILTRO[0]: LimitadorRPM(12), MODELO_EXTRACCION[0]: LimitadorRPM(8)}
contador_llamadas = Counter()

# Cortacircuitos: modelo -> instante (time.monotonic) hasta el que no se le envían peticiones
PAUSADOS: dict[str, float] = {}
PAUSA_SATURADO = 300   # 5 min: suficiente para que pase un pico de demanda

def llamar_llm(modelos, instrucciones, prompt, esquema, cache_key=None,
               max_output_tokens=16384, max_reintentos=4):
    """Structured output + rate limit + reintentos + iterador de modelos + caché en disco."""
    ruta = CACHE / f"{cache_key}.json" if cache_key else None
    if ruta and ruta.exists():
        return esquema.model_validate_json(ruta.read_text(encoding="utf-8"))

    config = types.GenerateContentConfig(
        system_instruction=instrucciones,
        temperature=0.0,
        max_output_tokens=max_output_tokens,
        response_mime_type="application/json",
        response_schema=esquema,
    )
    limitador = LIMITADORES[modelos[0]]
    # Cortacircuitos: se saltan los modelos marcados como saturados (si lo están todos, se prueban igualmente)
    ahora = time.monotonic()
    candidatos = [m for m in modelos if PAUSADOS.get(m, 0) <= ahora] or list(modelos)
    for modelo in candidatos:                   # iterador de modelos: si uno falla, se prueba el siguiente
        for intento in range(max_reintentos):
            limitador.esperar()
            try:
                resp = client.models.generate_content(model=modelo, contents=prompt, config=config)
                contador_llamadas[modelo] += 1
                resultado = resp.parsed if resp.parsed is not None else esquema.model_validate_json(resp.text)
                if ruta:
                    ruta.write_text(resultado.model_dump_json(), encoding="utf-8")
                return resultado
            except ValidationError:
                # JSON inválido o incompleto (p. ej. respuesta cortada al agotar max_output_tokens)
                if intento < max_reintentos - 1:
                    print(f"  [{modelo}] JSON inválido o truncado -> reintento {intento + 1}")
                else:
                    print(f"  [{modelo}] JSON inválido tras {max_reintentos} intentos -> siguiente modelo")
            except errors.APIError as e:
                if e.code == 404:                   # modelo retirado o no disponible para la cuenta
                    print(f"  [{modelo}] no disponible (404) -> se descarta; siguiente modelo")
                    PAUSADOS[modelo] = float("inf")
                    break
                if e.code in (429, 500, 503) and intento < max_reintentos - 1:
                    espera = 5 * 2 ** intento + random.uniform(0, 1)
                    print(f"  [{modelo}] error {e.code} -> reintento {intento + 1} en {espera:.0f}s")
                    time.sleep(espera)
                elif e.code in (429, 500, 503):     # reintentos agotados con este modelo
                    print(f"  [{modelo}] sigue fallando ({e.code}) -> en pausa {PAUSA_SATURADO // 60} min; siguiente modelo")
                    PAUSADOS[modelo] = time.monotonic() + PAUSA_SATURADO
                else:
                    raise
    raise RuntimeError(f"Ningún modelo respondió: {modelos}")

### Estimación de tokens y batching por presupuesto

Las reseñas del top 100 miden entre ~5.400 y ~8.000 caracteres. Para dimensionar los lotes se calibra la relación caracteres/token con `count_tokens`, que es gratuito y no consume cuota de generación. Se usa una **muestra aleatoria** y no las primeras filas: la fila 0 (`suiiii…`) tokeniza de forma anómala (3,79 caracteres/token frente a ~5,3 de una reseña normal) y sesgaría la estimación.

**Resultado (celda siguiente):** ~5 caracteres/token, **~1.360 tokens por reseña** (entre 1.100 y 1.600) y **~109.000 tokens** en total.

**Por qué por lotes, y de qué tamaño:**
- **La ventana de contexto no es la que limita.** Con 1M tokens de entrada, las 100 reseñas cabrían en una sola llamada. Lo que limita es lo demás: los **tokens de salida** por llamada (cada reseña genera su propio JSON), el **TPM** y la **calidad**, que baja en las últimas reseñas de un lote muy largo. A eso se suma que un lote que falla hay que repetirlo entero.
- **Por presupuesto de tokens, no por número fijo de filas.** Como el tamaño de las reseñas varía, cada lote se llena hasta un máximo de tokens de entrada, con un tope de filas. Así no hay que adivinar cuántas reseñas caben. Los límites de cada paso se justifican en su sección: ≤8.000 tokens y ≤15 filas en el filtro, ≤10.000 tokens y ≤5 filas en la extracción.
- **Ninguna reseña se pierde.** `ejecutar_por_lotes` comprueba que la respuesta traiga todos los `id` del lote. Si falta alguno, lo vuelve a pedir por separado.

In [9]:
muestra_cal = "\n".join(df_top["Contenido"].sample(10, random_state=42))
tokens_reales = client.models.count_tokens(model=MODELO_FILTRO[0], contents=muestra_cal).total_tokens
CHARS_POR_TOKEN = len(muestra_cal) / tokens_reales
print(f"Calibración: {CHARS_POR_TOKEN:.2f} caracteres/token")

def estimar_tokens(texto: str) -> int:
    return int(len(texto) / CHARS_POR_TOKEN) + 10   # +10 por delimitadores e id

def crear_lotes(textos: pd.Series, max_tokens: int, max_filas: int) -> list[list[int]]:
    lotes, actual, acumulado = [], [], 0
    for idx, texto in textos.items():
        t = estimar_tokens(texto)
        if actual and (acumulado + t > max_tokens or len(actual) >= max_filas):
            lotes.append(actual); actual, acumulado = [], 0
        actual.append(idx); acumulado += t
    if actual:
        lotes.append(actual)
    return lotes

df_top["tokens_est"] = df_top["Contenido"].map(estimar_tokens)
print("Tokens estimados por reseña:", df_top["tokens_est"].describe()[["min", "mean", "max"]].round(0).to_dict())
print("Tokens totales:", df_top["tokens_est"].sum())

def ejecutar_por_lotes(lotes, modelo, instrucciones, prompt_fn, esquema, prefijo):
    """Procesa lotes, valida que vuelvan todos los ids y reintenta individualmente los que falten."""
    salida = {}
    for n, ids in enumerate(lotes):
        res = llamar_llm(modelo, instrucciones, prompt_fn(ids), esquema, cache_key=f"{prefijo}_{n}")
        salida.update({r.id: r for r in res.resultados if r.id in ids})
        for i in set(ids) - set(salida):
            print(f"  id {i} omitido en lote {n}: reprocesando solo")
            res = llamar_llm(modelo, instrucciones, prompt_fn([i]), esquema, cache_key=f"{prefijo}_solo_{i}")
            salida.update({r.id: r for r in res.resultados if r.id == i})
        print(f"Lote {n + 1}/{len(lotes)} OK ({len(ids)} reseñas)")
    return pd.DataFrame([r.model_dump() for r in salida.values()]).set_index("id").sort_index()

Calibración: 4.99 caracteres/token
Tokens estimados por reseña: {'min': 1096.0, 'mean': 1360.0, 'max': 1607.0}
Tokens totales: 108829


---
## Paso 2 — LLM 1: filtrado de reseñas relevantes

### Criterio: un filtro permisivo a propósito

El objetivo del filtro es **no perder reseñas útiles** (priorizar el *recall*), no descartar mucho:
- Un **falso negativo** (una reseña útil descartada) se pierde para siempre: el paso 3 nunca la verá.
- Un **falso positivo** (una reseña poco útil que pasa) solo cuesta una extracción de más, y el paso 3 la trataría igualmente con valores "No mencionado".

Por eso el prompt termina con *"ante la duda, si se puede extraer al menos un aspecto concreto, marca relevante=true"*. Además, tras el deduplicado las 80 reseñas son textos de 5.400 a 8.000 caracteres, casi todos análisis reales: lo esperable es que el filtro descarte pocas. **Resultado: 79 de 80 relevantes**; la única descartada es el texto de relleno `suiiii…` (ver *Auditoría del filtro*).

### Diseño del prompt

| Técnica | Cómo se aplica | Para qué |
|---|---|---|
| **Mensaje *system*** con rol y reglas | Las instrucciones van en `system_instruction`; cada lote solo lleva las reseñas | Separa el comportamiento global de los datos de cada llamada |
| **Criterio operativo** | "Relevante" = opinión sobre el juego argumentada con al menos un aspecto concreto | Es exactamente lo que el paso 3 necesita para extraer algo |
| **Aviso sobre el formato** | Se avisa de que el texto viene en minúsculas, sin puntuación y sin palabras vacías | Evitar el riesgo de que el modelo descarte reseñas válidas por parecer "mal escritas" |
| **Lista de casos irrelevantes** | Relleno, repeticiones, texto ajeno al juego, notas sin argumentos | Casos que aparecen realmente en este dataset |
| ***Few-shot* con casos frontera** | 5 ejemplos en el mismo estilo que los datos, incluido uno frontera real: reseñas que empiezan con autopromoción (`follow our curator page…`) pero luego analizan el juego, y que **sí** son relevantes | Fijar el umbral de decisión justo donde el modelo podría dudar |
| ***Structured output*** | Esquema Pydantic: `id`, `relevante` (booleano) y `motivo` | La decisión es un booleano, no texto que haya que interpretar |
| **Temperatura 0** | Igual en todas las llamadas | Clasificación con salidas discretas: se busca determinismo, no creatividad |
| **Campo `motivo`** | Justificación de máx. 15 palabras | Registro para **auditar** cada decisión. Va después de `relevante` en el esquema, así que el modelo primero decide y luego justifica: no es un razonamiento previo (*chain-of-thought*) |

### Qué texto se envía: inicio + final

Para decidir si una reseña es informativa no hace falta leerla entera. Se envían los **primeros 1.200 y los últimos 800 caracteres**: ~400 tokens en lugar de ~1.360 de media (calibración: 4,99 caracteres por token).
- El inicio muestra de qué va la reseña.
- El final detecta reseñas que degeneran en relleno o repeticiones, y suele contener el veredicto ("overall…", "in conclusion…").

### Tamaño de lote

**Máximo de 15 reseñas y ~8.000 tokens de entrada por lote.** En la práctica, lo que limita es el número de filas: 15 extractos suman ~6.100 tokens. La salida es mínima (~30 tokens por reseña), así que no hay riesgo de que se corte. Resultado: **6 llamadas en lugar de 80**, en menos de un minuto.

No se envían las 80 en una sola llamada, aunque cabrían en el contexto: con listas largas aumenta el riesgo de que el modelo omita filas o relaje el criterio hacia el final. Si aun así omite alguna, el código lo detecta (validación de ids) y la reprocesa por separado.

In [10]:
CHARS_INICIO, CHARS_FINAL = 1200, 800

def extracto(texto: str) -> str:
    if len(texto) <= CHARS_INICIO + CHARS_FINAL:
        return texto
    return f"{texto[:CHARS_INICIO]} [...] {texto[-CHARS_FINAL:]}"

class DecisionFiltro(BaseModel):
    id: int
    relevante: bool
    motivo: str = Field(description="Justificación en máx. 15 palabras, en español")

class LoteFiltro(BaseModel):
    resultados: list[DecisionFiltro]

INSTRUCCIONES_FILTRO = """Eres un moderador experto en reseñas de videojuegos de Steam. Tu única tarea es decidir
si cada reseña es RELEVANTE para un análisis posterior de opinión.

FORMATO DEL TEXTO: las reseñas están preprocesadas. Todo en minúsculas, sin signos de puntuación,
a veces con palabras pegadas, y en español sin palabras vacías (artículos, preposiciones).
Pueden estar en inglés o en español. NO las descartes por estar mal escritas.
Recibes el inicio y el final de cada reseña separados por [...].

RELEVANTE = contiene una opinión sobre el juego argumentada con al menos un aspecto concreto
(jugabilidad, historia, personajes, gráficos, sonido, dificultad, rendimiento o bugs, precio,
duración, contenido, mundo abierto, combate, controles...).

IRRELEVANTE si la reseña es mayoritariamente:
- Relleno: letras o palabras repetidas sin sentido.
- El mismo texto o frase repetido una y otra vez.
- Texto sin relación con el juego (chistes, letras de canciones, historias personales sin valorar el juego).
- Valoraciones sin argumento ("10 de 10", "masterpiece", "basura") aunque sean largas.
- Quejas sobre la tienda, la empresa o terceros sin valorar el propio juego.

Ejemplos:
<resena id="900">suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii</resena>
-> {"id": 900, "relevante": false, "motivo": "Texto de relleno sin contenido"}
<resena id="901">the combat is deep and every boss forces you to learn patterns but the camera is awful in tight spaces and performance drops in the open world</resena>
-> {"id": 901, "relevante": true, "motivo": "Valora combate, jefes, cámara y rendimiento"}
<resena id="902">if youre looking for more quality reviews like this follow our curator page night city looks stunning but the main quest is short and bugs ruined several missions</resena>
-> {"id": 902, "relevante": true, "motivo": "Pese a la autopromoción valora ciudad, historia y bugs"}
<resena id="903">best game ever best game ever best game ever 10 de 10 10 de 10 goty goty goty</resena>
-> {"id": 903, "relevante": false, "motivo": "Repeticiones y notas sin ningún argumento"}
<resena id="904">historia preciosa personajes muy trabajados pero precio excesivo duracion corta</resena>
-> {"id": 904, "relevante": true, "motivo": "Valora historia, personajes, precio y duración"}

Ante la duda, si se puede extraer al menos un aspecto positivo o negativo concreto, marca relevante=true.
Devuelve exactamente un resultado por cada reseña recibida, con su mismo id."""

def prompt_filtro(ids: list[int]) -> str:
    bloques = [f'<resena id="{i}">{extracto(df_top.at[i, "Contenido"])}</resena>' for i in ids]
    return "Clasifica las siguientes reseñas:\n\n" + "\n".join(bloques)

print(prompt_filtro([0])[:400])

Clasifica las siguientes reseñas:

<resena id="0">suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisui


In [11]:
lotes_filtro = crear_lotes(df_top["Contenido"].map(extracto), max_tokens=8000, max_filas=15)
print(f"{len(lotes_filtro)} lotes -> tamaños {[len(l) for l in lotes_filtro]}")

decisiones = ejecutar_por_lotes(lotes_filtro, MODELO_FILTRO, INSTRUCCIONES_FILTRO,
                                prompt_filtro, LoteFiltro, "filtro")
df_top = df_top.join(decisiones)
df_top["relevante"] = df_top["relevante"].fillna(False).astype(bool)

df_relevantes = df_top[df_top["relevante"]].copy()
print(f"\nRelevantes: {len(df_relevantes)} / {len(df_top)}")
print(df_relevantes["Valoración"].value_counts())

6 lotes -> tamaños [15, 15, 15, 15, 15, 5]
Lote 1/6 OK (15 reseñas)
Lote 2/6 OK (15 reseñas)
Lote 3/6 OK (15 reseñas)
Lote 4/6 OK (15 reseñas)
Lote 5/6 OK (15 reseñas)
Lote 6/6 OK (5 reseñas)



Relevantes: 79 / 80
Valoración
No recomendado    49
Recomendado       30
Name: count, dtype: int64


### Auditoría del filtro

Se revisan los dos tipos de error posibles:

- **Descartes (falsos negativos):** solo se descarta 1 reseña de 80, la de relleno `suiiii…` (tabla siguiente). Es correcto, pero hay que matizarlo: esa reseña es prácticamente el ejemplo *few-shot* 900 del prompt, así que acertar en ella no demuestra gran cosa sobre la capacidad del filtro.
- **Aceptadas (falsos positivos):** se comprueban con el resultado de la extracción (celda de validación de *Resultados*). Ninguna de las 79 reseñas relevantes sale sin aspectos (mínimo 5, mediana 8), ninguna tiene todas las notas a "No mencionado" y la más corta tiene más de 5.000 caracteres. Las 3 reseñas con autopromoción de *curator* (el caso frontera del ejemplo 902) pasan correctamente, porque aun así valoran el juego.

**¿Para qué sirve un filtro que descarta 1 de 80?** El top 100 de la plantilla ya prioriza reseñas largas, así que llega poco ruido. El filtro es una **salvaguarda barata** (6 llamadas al modelo *lite*, con extractos de cada reseña) que se justifica al aplicar el *pipeline* a otra muestra. Además, deja un `motivo` por reseña que permite auditar cada decisión. Como se prioriza no perder reseñas útiles (*recall*), si apareciera un falso negativo se corregiría añadiéndolo al prompt como ejemplo *few-shot*.

In [12]:
display(df_top.loc[~df_top["relevante"], ["Contenido", "motivo"]].assign(Contenido=lambda d: d["Contenido"].str[:200]))
display(df_top.loc[df_top["relevante"], ["Contenido", "motivo"]].head(5).assign(Contenido=lambda d: d["Contenido"].str[:120]))

,Contenido,motivo
id,,
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiii...,Texto de relleno sin contenido


,Contenido,motivo
id,,
1,this was probably my first preorder i felt that if the game turned out to be the worst game i played or didnt even run 1,"Valora diseño urbano, historia, inmersión y rol"
2,oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty bia,"Valora narrativa, misiones, conducción y bugs"
3,i know many will handwave away any criticisms of this game as salt or reply with git gud but as fan of fromsoft games si,"Valora dificultad, combate, jefes y diseño de enemigos"
4,i had enormous expectations having put thousand hours into dark souls 3 alone sadly er is tainted by design decisions no,"Valora combate, gráficos, rendimiento y mundo abierto"
5,i finished the witcher 3 for the first time this year and because of the interesting story told i got invested and start,"Valora caminos de vida, builds, música y jugabilidad"


---
## Paso 3 — LLM 2: extracción de información estructurada

### Esquema de salida

Combina el esquema orientativo del enunciado con las entidades del CSV de ejemplo (`analisis_videojuegos_resultados.csv`), y añade campos de justificación:

| Campo | Tipo | Origen / justificación |
|---|---|---|
| `sentimiento_general` | Positivo / Negativo / Neutral / Mixto | Enunciado. "Mixto" es muy frecuente en reseñas largas ("me encanta pero…") |
| `aspectos_juego` | objeto con una nota por aspecto (jugabilidad, gráficos, historia, sonido, controles, rendimiento) | CSV de ejemplo (`game_aspects`), pero con escala cerrada |
| `aspectos_positivos` / `aspectos_negativos` | listas de etiquetas canónicas | Enunciado. Con vocabulario fijo se pueden contar las etiquetas |
| `problemas_tecnicos` | lista de bugs o fallos concretos | CSV de ejemplo (`technical_issues`). Muy frecuente en este dataset (Cyberpunk 2077…) |
| `dificultad` | 5 niveles + No mencionado | Enunciado |
| `relacion_calidad_precio` | Alta / Media / Baja / No mencionado | CSV de ejemplo (`value_for_money`) |
| `horas_jugadas` | entero o nulo | CSV de ejemplo (`playtime_mentioned`). Cuánto ha jugado el autor da credibilidad a su opinión |
| `recomendaria` | Sí / No / Condicional | CSV de ejemplo (`would_recommend`). El matiz "condicional" no existe en la etiqueta binaria |
| `publico_objetivo` | texto corto | CSV de ejemplo (`target_audience`) |
| `evidencia` | 1-3 citas literales | **Justificación**: cada valoración se puede rastrear al texto |
| `resumen` | una frase en español | CSV de ejemplo (`summary`) |

El `Recomendado` del enunciado **no se pide al LLM**: viene del dataset (`Valoración`, `Recomendado_binario`) y se une a la extracción por el índice del DataFrame (el `id` de cada reseña). Así sirve como **verdad de referencia** para validar el `sentimiento_general` y el `recomendaria` que extrae el modelo.

### Diseño del prompt

| Técnica | Cómo se aplica | Para qué |
|---|---|---|
| **Mensaje *system*** con rol | "Analista de producto de una empresa de videojuegos" que busca fortalezas y problemas | Orienta la extracción hacia información accionable, no hacia un resumen genérico |
| **Aviso sobre el formato** | Texto en minúsculas, sin puntuación y sin palabras vacías; se pide "reconstruir el sentido de las frases antes de analizarlas" | Que el modelo interprete bien un texto difícil de leer |
| **Regla "no deducir"** | Si un atributo no aparece: "No mencionado" (o `null` en `horas_jugadas`) | Separar **ausencia de información** de **opinión neutra**, y evitar alucinaciones |
| **Vocabulario canónico** | 26 etiquetas preferentes para los aspectos; solo se crea una nueva si ninguna encaja | Que "gráficos", "graphics" y "apartado visual" cuenten como lo mismo al agregar |
| **Criterios para los casos ambiguos** | Definición explícita de "Mixto" y de "Condicional", y regla contra la ironía, cada una con un miniejemplo dentro de la propia regla | Resolver los casos frontera de esta tarea |
| **Zero-shot con reglas, no *few-shot*** | Sin ejemplos completos de reseña → JSON | Un ejemplo útil exigiría una reseña entera (~1.360 tokens) más su JSON en **cada** lote. El formato ya lo garantiza el esquema, y los matices, las reglas con miniejemplos |
| ***Structured output*** | Esquema Pydantic con valores cerrados (`Literal`), objeto anidado y campo nulo | JSON siempre válido y valores agregables. Es lo que falla en el CSV de ejemplo (`neutro`, `neutral`, `mixed`…) |
| **Temperatura 0** | Igual en todas las llamadas | Extracción fiel al texto y reproducible |
| **Citas literales (`evidencia`)** | Fragmentos copiados del texto original, en su idioma | Que cada valoración se pueda comprobar |

**Comprobación de las citas** (calculada en la celda de validación de *Resultados*): el **97 % de las citas son literales** (229 de 237), es decir, aparecen tal cual en la reseña; en 71 de las 79 reseñas lo son todas. Las 8 restantes son casi literales: el modelo separa palabras pegadas, corrige la concordancia, añade signos de puntuación o desarrolla abreviaturas (`cp2077` → `cyberpunk 2077`), y en 2 casos une dos fragmentos separados del texto. La trazabilidad es alta, pero no perfecta.

In [13]:
Nota = Literal["Excelente", "Buena", "Regular", "Mala", "No mencionado"]

class AspectosJuego(BaseModel):
    jugabilidad: Nota
    graficos: Nota
    historia: Nota
    sonido: Nota
    controles: Nota
    rendimiento: Nota = Field(description="Optimización, estabilidad, bugs")

class ExtraccionResena(BaseModel):
    id: int
    sentimiento_general: Literal["Positivo", "Negativo", "Neutral", "Mixto"]
    aspectos_juego: AspectosJuego
    aspectos_positivos: list[str] = Field(description="Etiquetas canónicas, máx. 5")
    aspectos_negativos: list[str] = Field(description="Etiquetas canónicas, máx. 5")
    problemas_tecnicos: list[str] = Field(description="Bugs o fallos concretos citados, en español, máx. 5")
    dificultad: Literal["Demasiado fácil", "Fácil", "Equilibrado", "Difícil", "Demasiado difícil", "No mencionado"]
    relacion_calidad_precio: Literal["Alta", "Media", "Baja", "No mencionado"]
    horas_jugadas: Optional[int] = Field(description="Horas que el autor dice haber jugado; null si no lo dice")
    recomendaria: Literal["Sí", "No", "Condicional"]
    publico_objetivo: str = Field(description="A quién lo recomienda el autor, máx. 10 palabras, o 'No mencionado'")
    evidencia: list[str] = Field(description="1-3 fragmentos literales del texto (máx. 20 palabras cada uno)")
    resumen: str = Field(description="Una frase en español")

class LoteExtraccion(BaseModel):
    resultados: list[ExtraccionResena]

ETIQUETAS = ["jugabilidad", "historia", "personajes", "gráficos", "dirección artística", "banda sonora",
             "dificultad", "rendimiento", "bugs", "controles", "duración", "rejugabilidad", "precio",
             "multijugador", "contenido", "mundo abierto", "exploración", "combate", "progresión",
             "conducción", "IA de NPCs", "misiones secundarias", "decisiones del jugador", "mods",
             "microtransacciones", "marketing engañoso"]

INSTRUCCIONES_EXTRACCION = f"""Eres un analista de producto de una empresa de videojuegos. Extraes información
estructurada de reseñas de jugadores de Steam para detectar fortalezas y problemas de cada juego.

FORMATO DEL TEXTO: las reseñas están preprocesadas. Todo en minúsculas, sin puntuación, a veces con
palabras pegadas, y en español sin palabras vacías. Pueden estar en inglés o en español.
Reconstruye mentalmente el sentido de las frases antes de analizarlas.

Reglas:
1. Básate SOLO en el texto. Si un atributo no se menciona, usa "No mencionado" (o null en horas_jugadas). No lo deduzcas.
2. aspectos_positivos / aspectos_negativos: usa preferentemente estas etiquetas canónicas:
   {", ".join(ETIQUETAS)}.
   Crea una etiqueta nueva (1-3 palabras, minúsculas, en español) solo si ninguna encaja.
   Un aspecto puede aparecer en las dos listas si el autor lo matiza.
3. Detecta ironía y sarcasmo: "great another patch that breaks the game" es negativo.
4. sentimiento_general = "Mixto" si hay elogios y críticas de peso similar.
5. recomendaria = "Condicional" si el autor recomienda con condiciones ("only on sale", "wait for patches", "solo si te gustan los rpg").
6. evidencia: copia fragmentos LITERALES del texto original, en su idioma, tal como aparecen.
7. Todos los demás campos de texto, en español.
8. Devuelve un resultado por cada reseña recibida, con su mismo id."""

def prompt_extraccion(ids: list[int]) -> str:
    bloques = [f'<resena id="{i}">\n{df_relevantes.at[i, "Contenido"]}\n</resena>' for i in ids]
    return "Analiza las siguientes reseñas:\n\n" + "\n\n".join(bloques)

### Tamaño de lote del paso 3

Aquí las reseñas se envían **completas**, porque cada matiz cuenta: ~1.360 tokens de entrada por reseña (ver *Estimación de tokens*).

- **Lote: máx. 5 reseñas / 10.000 tokens de entrada.** En la práctica limita el número de filas: 5 reseñas, incluso de las más largas (5 × ~1.600), no llegan a los 10.000 tokens. Cada lote queda en ~7.000 tokens de entrada.
- **Salida medida** (con `count_tokens` sobre las respuestas guardadas): **~270 tokens por reseña** y entre 1.100 y 1.500 por lote. Es muy poco frente a `max_output_tokens=16384`, pero el margen hace falta: los modelos *flash* gastan también tokens de **razonamiento** interno, que cuentan dentro de ese límite y no aparecen en la respuesta. **Con el límite inicial de 8.192, `gemini-flash-latest` devolvió en un lote un JSON cortado a la mitad.** Por eso se duplicó el límite y, además, `llamar_llm` trata un JSON inválido o incompleto como error temporal: lo reintenta y, si persiste, pasa al siguiente modelo.
- **Cuotas:** ~7.000 tokens por llamada a un máximo de 8 RPM son ~56.000 TPM, muy por debajo del 1M de TPM; 16 llamadas tampoco se acercan a las 1.500 RPD. Ninguna cuota nominal limita el tamaño del lote.
- **Lotes más pequeños:** más llamadas, y por tanto más exposición a los 429 por **saturación del servicio**, que aparecen aunque no se alcancen las cuotas nominales (lote 3 de la ejecución).
- **Lotes más grandes:** más tokens de salida y de razonamiento por llamada (riesgo de JSON truncado), peor calidad en las últimas reseñas del lote y más trabajo perdido si una llamada falla.

**Resultado:** 79 reseñas relevantes → **16 llamadas** (15 lotes de 5 y uno de 4). El limitador impone un mínimo de ~2 minutos; el tiempo real fue mayor por la latencia y los reintentos (ver *Gestión de limitaciones*).

In [14]:
lotes_ext = crear_lotes(df_relevantes["Contenido"], max_tokens=10000, max_filas=5)
print(f"{len(lotes_ext)} lotes -> tamaños {[len(l) for l in lotes_ext]}")

extraccion = ejecutar_por_lotes(lotes_ext, MODELO_EXTRACCION, INSTRUCCIONES_EXTRACCION,
                                prompt_extraccion, LoteExtraccion, "extraccion")

# Aplanar el objeto anidado aspectos_juego en columnas
aspectos_df = pd.json_normalize(extraccion["aspectos_juego"].tolist()).set_index(extraccion.index).add_prefix("nota_")
extraccion = extraccion.drop(columns="aspectos_juego").join(aspectos_df)

df_final = df_relevantes[["indice_original", "Contenido", "Valoración", "Recomendado_binario"]].join(extraccion)
df_final.head()

16 lotes -> tamaños [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4]


Lote 1/16 OK (5 reseñas)


Lote 2/16 OK (5 reseñas)


  [gemini-3-flash-preview] error 429 -> reintento 1 en 5s


  [gemini-3-flash-preview] error 429 -> reintento 2 en 11s


  [gemini-3-flash-preview] error 429 -> reintento 3 en 21s


... (3 outputs left out)

  [gemini-flash-latest] error 429 -> reintento 3 en 21s


  [gemini-flash-latest] sigue fallando (429) -> en pausa 5 min; siguiente modelo


Lote 3/16 OK (5 reseñas)
Lote 4/16 OK (5 reseñas)
Lote 5/16 OK (5 reseñas)
Lote 6/16 OK (5 reseñas)
Lote 7/16 OK (5 reseñas)
Lote 8/16 OK (5 reseñas)
Lote 9/16 OK (5 reseñas)
Lote 10/16 OK (5 reseñas)
Lote 11/16 OK (5 reseñas)
Lote 12/16 OK (5 reseñas)
Lote 13/16 OK (5 reseñas)
Lote 14/16 OK (5 reseñas)
Lote 15/16 OK (5 reseñas)
Lote 16/16 OK (4 reseñas)


,indice_original,Contenido,Valoración,Recomendado_binario,sentimiento_general,aspectos_positivos,aspectos_negativos,problemas_tecnicos,dificultad,relacion_calidad_precio,...,recomendaria,publico_objetivo,evidencia,resumen,nota_jugabilidad,nota_graficos,nota_historia,nota_sonido,nota_controles,nota_rendimiento
id,,,,,,,,,,,,,,,,,,,,,
1,17660,this was probably my first preorder i felt that if the game turned out to be the worst game i played or didnt even run 10 dollars i paid for the w...,Recomendado,1,Positivo,"[mundo abierto, exploración, historia, personajes, jugabilidad]",[dificultad],[],Equilibrado,Alta,...,Sí,Amantes de la exploración urbana y RPGs con historias profundas,"[the greatest city that ever been built in game, beautiful lovechild of the witcher and deus ex, story and the characters are amazing]",El jugador considera que es la mejor representación de una ciudad en un videojuego y alaba su profundidad narrativa.,Excelente,Excelente,Excelente,No mencionado,No mencionado,Buena
2,4187,oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and the...,No recomendado,0,Negativo,"[dirección artística, banda sonora, mundo abierto]","[bugs, contenido, misiones secundarias, rendimiento, personalización]","[glitches gráficos, texturas faltantes, coches desapareciendo, objetos no saqueables, misiones que no terminan]",No mencionado,Baja,...,No,No mencionado,"[cp 2077 is very dissapointing by cdpr standards, bugs became more frequent and severe as i played more, cyberpunk 2077 feels unfinished rushed bu...",Un fan de CDPR se siente decepcionado por el estado técnico y la falta de mecánicas básicas de rol.,Regular,Buena,Regular,Excelente,Buena,Mala
3,5415,i know many will handwave away any criticisms of this game as salt or reply with git gud but as fan of fromsoft games since demons souls this game...,No recomendado,0,Mixto,"[dirección artística, historia, personajes, mods, mundo abierto]","[dificultad, rendimiento, bugs, contenido]","[problemas de rendimiento en PC, barras de vida de enemigos se rellenan, muerte súbita de la montura]",Demasiado difícil,No mencionado,...,Condicional,Jugadores dispuestos a usar mods cooperativos o builds específicas,"[most punitive and least satisfying of the games, boss design is in my opinion some of the worst, seamless coop mod allowed something genuinely ma...","Aunque visualmente bello, el autor encuentra el diseño de jefes frustrante, mejorando su opinión gracias a mods.",Regular,Excelente,Buena,No mencionado,No mencionado,Regular
4,7751,i had enormous expectations having put thousand hours into dark souls 3 alone sadly er is tainted by design decisions not fully thought through an...,No recomendado,0,Negativo,"[jugabilidad, dirección artística]","[rendimiento, historia, misiones secundarias, multijugador, bugs]","[stuttering por compilación de shaders, crasheos, tramposos en multijugador, latencia en registro de golpes]",No mencionado,No mencionado,...,No,Jugadores que solo busquen combate brutal sin profundidad narrativa,"[er is tainted by design decisions not fully thought through, story is shockingly conservative with almost identical plot points, worst shader com...",El autor critica duramente la falta de innovación narrativa y los graves problemas técnicos de la versión de PC.,Regular,Regular,Mala,Buena,Buena,Mala
5,16503,i finished the witcher 3 for the first time this year and because of the interesting story told i got invested and started reading about the lore ...,Recomendado,1,Mixto,"[historia, personajes, banda sonora, mundo abierto, progresión]","[IA de NPCs, bugs, marketing engañoso, contenido]","[desaparición de coches a distancia, sistema de policía deficiente, IA de ciudadanos inexistente]",No mencionado,No mencionado,...,Sí,Fans de historias inmersivas y personajes bien escritos,"[story was worth experiencing, characters feel like they have their own ambitions, general npc ai is barely 

---
## Resultados e insights

### Validación: ¿lo extraído concuerda con la recomendación real?

`Valoración` es la etiqueta real que puso el autor y **no se pasa al LLM**, así que sirve como verdad de referencia. Si la extracción es buena, `Positivo` y `recomendaria = Sí` deben caer en *Recomendado*, y `Negativo` y `No` en *No recomendado*. `Mixto` y `Condicional` recogen los matices que la etiqueta binaria no tiene.

**Resultado (celda siguiente):**
- **Ningún error cruzado.** Las 16 reseñas `Positivo` son *Recomendado* y las 29 `Negativo` son *No recomendado* (100 % en las 45 de sentimiento claro). Lo mismo con `recomendaria`: 22 `Sí` y 39 `No`, todas en su clase.
- **El matiz importa.** El 43 % de las reseñas (34) son `Mixto` y 18 son `Condicional`. De las `Mixto`, 20 acaban en *No recomendado*: al jugador le gusta el juego, pero no lo recomienda en su estado actual. La etiqueta binaria esconde esa información.
- **Trazabilidad.** El 97 % de las citas de `evidencia` son literales (detalle en *Paso 3*).

In [15]:
display(pd.crosstab(df_final["sentimiento_general"], df_final["Valoración"], margins=True))
display(pd.crosstab(df_final["recomendaria"], df_final["Valoración"], margins=True))

polar = df_final[df_final["sentimiento_general"].isin(["Positivo", "Negativo"])]
acierto = ((polar["sentimiento_general"] == "Positivo") == (polar["Recomendado_binario"] == 1)).mean()
print(f"Concordancia sentimiento polar vs. recomendación real: {acierto:.1%} ({len(polar)} reseñas)")

# Trazabilidad: ¿las citas de `evidencia` aparecen tal cual en el texto de la reseña?
pares = list(zip(df_final["evidencia"], df_final["Contenido"]))
citas = [(c.strip(), texto) for lista, texto in pares for c in lista]
literales = sum(c in texto for c, texto in citas)
todas = sum(all(c.strip() in texto for c in lista) for lista, texto in pares)
print(f"Citas literales: {literales}/{len(citas)} ({literales / len(citas):.0%}) | "
      f"reseñas con todas sus citas literales: {todas}/{len(df_final)}")

# Auditoría del filtro: ¿alguna reseña aceptada como relevante quedó sin contenido analizable?
n_aspectos = df_final["aspectos_positivos"].map(len) + df_final["aspectos_negativos"].map(len)
sin_notas = (df_final.filter(like="nota_") == "No mencionado").all(axis=1).sum()
print(f"Aspectos por reseña relevante: mínimo {n_aspectos.min()}, mediana {n_aspectos.median():.0f} | "
      f"sin ningún aspecto: {(n_aspectos == 0).sum()} | con todas las notas 'No mencionado': {sin_notas}")

Valoración,No recomendado,Recomendado,All
sentimiento_general,,,
Mixto,20,14,34
Negativo,29,0,29
Positivo,0,16,16
All,49,30,79


Valoración,No recomendado,Recomendado,All
recomendaria,,,
Condicional,10,8,18
No,39,0,39
Sí,0,22,22
All,49,30,79


Concordancia sentimiento polar vs. recomendación real: 100.0% (45 reseñas)
Citas literales: 229/237 (97%) | reseñas con todas sus citas literales: 71/79
Aspectos por reseña relevante: mínimo 5, mediana 8 | sin ningún aspecto: 0 | con todas las notas 'No mencionado': 0


In [16]:
def contar(serie_listas):
    """Nº de reseñas que mencionan cada etiqueta (normalizada a minúsculas)."""
    return pd.Series(Counter(a.strip().lower() for l in serie_listas for a in l), dtype=int)

# Conteo completo de cada columna: si una etiqueta no aparece en una columna, su 0 es real
conteos = pd.DataFrame({
    "positivos (todas)": contar(df_final["aspectos_positivos"]),
    "negativos (todas)": contar(df_final["aspectos_negativos"]),
    "negativos (No recomendado)": contar(df_final.loc[df_final["Recomendado_binario"] == 0, "aspectos_negativos"]),
}).fillna(0).astype(int)

# Filas: unión de los 10 más citados de cada columna, ordenadas por su máximo
filas = set().union(*(conteos[c].nlargest(10).index for c in conteos))
display(conteos.loc[sorted(filas, key=lambda f: (-conteos.loc[f].max(), f))])

display(contar(df_final["problemas_tecnicos"]).sort_values(ascending=False).head(10)
        .rename("problemas técnicos más citados"))

,positivos (todas),negativos (todas),negativos (No recomendado)
bugs,0,50,30
historia,40,11,8
rendimiento,1,35,25
dirección artística,34,0,0
banda sonora,33,1,0
ia de npcs,0,33,25
personajes,32,1,0
mundo abierto,30,16,13
contenido,8,23,15
gráficos,22,2,1


bugs                      4
t-posing                  4
bugs visuales             3
objetos flotantes         3
glitches gráficos         3
caídas de frames          3
crashes                   2
glitches                  2
texturas que no cargan    2
caídas de fps             2
Name: problemas técnicos más citados, dtype: int64

In [17]:
cols_nota = [c for c in df_final.columns if c.startswith("nota_")]
display(df_final[cols_nota].apply(pd.Series.value_counts).fillna(0).astype(int)
        .reindex(["Excelente", "Buena", "Regular", "Mala", "No mencionado"]))

for col in ["dificultad", "relacion_calidad_precio"]:
    display(pd.crosstab(df_final[col], df_final["Valoración"]))

print("Horas jugadas mencionadas:", df_final["horas_jugadas"].notna().sum(),
      "| mediana:", df_final["horas_jugadas"].median())

,nota_jugabilidad,nota_graficos,nota_historia,nota_sonido,nota_controles,nota_rendimiento
Excelente,11,43,19,26,3,0
Buena,16,16,28,8,7,8
Regular,37,6,17,4,12,16
Mala,15,3,8,5,19,40
No mencionado,0,11,7,36,38,15


Valoración,No recomendado,Recomendado
dificultad,,
Demasiado difícil,3,0
Demasiado fácil,1,0
Difícil,7,5
Equilibrado,4,9
Fácil,5,4
No mencionado,29,12


Valoración,No recomendado,Recomendado
relacion_calidad_precio,,
Alta,0,11
Baja,17,0
Media,2,3
No mencionado,30,16


Horas jugadas mencionadas: 41 | mediana: 54.0


### Qué dicen las reseñas

Lectura de las dos celdas anteriores (aspectos citados, notas por aspecto, dificultad, calidad-precio y horas jugadas):

- **Lo artístico gusta y lo técnico hunde la valoración.** Los aspectos positivos más citados son la historia (40), la dirección artística (34), la banda sonora (33) y los personajes (32); los tres últimos casi no aparecen como negativos. En el lado negativo dominan los bugs (50), el rendimiento (35) y la IA de los NPCs (33). La nota de rendimiento es `Mala` en 40 reseñas y `Excelente` en ninguna; la de gráficos es `Excelente` en 43.
- **Los fallos técnicos no siempre impiden recomendar.** 20 de las 50 reseñas que se quejan de bugs recomiendan el juego igualmente. Las quejas por rendimiento e IA, en cambio, se concentran en *No recomendado* (25 de 35 y 25 de 33).
- **Aspectos de doble filo.** El mundo abierto (30 positivos / 16 negativos), las misiones secundarias (13 / 19), el combate (17 / 14) y la jugabilidad (11 / 12) dividen a los jugadores: atraen por la ambición y decepcionan por la ejecución. El contenido y la progresión son sobre todo negativos (23 y 15).
- **Expectativas incumplidas.** 17 reseñas hablan de *marketing* engañoso, 7 de ellas aun recomendando el juego: la decepción respecto a lo prometido pesa en la valoración.
- **El precio va de la mano de la recomendación.** Las 11 reseñas con calidad-precio `Alta` recomiendan el juego y las 17 con `Baja` no lo hacen.
- **Opiniones con experiencia.** 41 reseñas mencionan las horas jugadas (mediana: 54 h), así que las críticas no vienen de partidas cortas.

**Insight accionable:** la prioridad de mejora es técnica (estabilidad, rendimiento e IA de los NPCs) y de profundidad del mundo abierto. Lo narrativo y lo artístico son el punto fuerte del juego y no necesitan cambios.

**Limitaciones:**
- `problemas_tecnicos` es texto libre sin vocabulario cerrado y sale muy fragmentado (`bugs`, `glitches`, `glitches gráficos`…). Para agregarlo habría que normalizarlo con etiquetas canónicas, igual que los aspectos.
- La muestra es pequeña (79 reseñas) y está sesgada hacia *No recomendado* (49 frente a 30): las cifras describen esta muestra, no al conjunto de jugadores.

### Exportación

Se guarda con otro nombre para **no sobrescribir** el CSV de ejemplo de la plantilla (`analisis_videojuegos_resultados.csv`). Las listas se serializan como JSON para que el CSV se pueda volver a leer sin perder estructura.

In [18]:
salida = df_final.copy()
for col in ["aspectos_positivos", "aspectos_negativos", "problemas_tecnicos", "evidencia"]:
    salida[col] = salida[col].map(lambda l: json.dumps(l, ensure_ascii=False))
salida.to_csv("resultados_analisis_videojuegos.csv", encoding="utf-8-sig")
print("Filas exportadas:", len(salida))
print("Llamadas realizadas:", dict(contador_llamadas))

Filas exportadas: 79
Llamadas realizadas: {'gemini-3-flash-preview': 2, 'gemini-3.5-flash-lite': 1}


---
## Resumen de decisiones técnicas

| Decisión | Valor | Justificación |
|---|---|---|
| Preselección | Top 100 de la plantilla + deduplicado exacto (80 reseñas únicas) | Pandas para lo determinista y gratuito; el LLM solo para lo semántico |
| Formato del texto | Avisado en ambos prompts; ejemplos *few-shot* en el mismo estilo | El texto viene en minúsculas, sin puntuación y sin palabras vacías |
| LLM 1 · Filtro | `gemini-3.5-flash-lite`, T=0; inicio + final de cada reseña; lotes de ≤8k tokens y ≤15 filas | Clasificación sencilla: modelo barato y rápido; ~70 % menos tokens; 6 llamadas |
| LLM 2 · Extracción | `gemini-3-flash-preview`, T=0; texto completo; lotes de ≤10k tokens y ≤5 filas | Extracción con matices; el tamaño de lote lo limitan los tokens de salida |
| Justificación | `motivo` en el filtro y `evidencia` literal en la extracción | Decisiones auditables |
| Validación | Cruce de `sentimiento_general` y `recomendaria` con la `Valoración` real | Métrica objetiva: 100 % de concordancia en las 45 reseñas de sentimiento claro |

- **Modelos, disponibilidad real y extracción no homogénea:** ver *Proveedor y modelos*.
- **Rate limits, reintentos, llamada multimodelo, cortacircuitos y caché:** ver *Gestión de limitaciones*.

**Volumen:** 22 lotes (6 de filtro + 16 de extracción) en lugar de ~160 llamadas reseña a reseña. A esos lotes hay que sumar los reintentos causados por la saturación del servicio.